# Détection satellite — inondation Émilie-Romagne (EMSR664, mai 2023)

**Événement réel, terminé (`CLOSED`)** : activation Copernicus EMS EMSR664, crue du 16/05/2023 en Émilie-Romagne (Italie), suite à une première vague le 02/05/2023. 3 morts, centaines de déplacés, 34 cartes officielles publiées par Copernicus EMS. Rivières concernées : Idice, Samoggia, Savio, Marzeno, Voltre, Marecchia, Pisciatello, Ausa, Montone.

**Pourquoi cet événement plutôt que EMSR926 (Lettonie)** : événement clos, le satellite a eu largement le temps de repasser après l'inondation — pas de blocage "en attente du prochain passage" comme sur le cas précédent. Même méthode, même code : c'est la vraie démonstration que ce qu'on a construit sur la Lettonie n'était pas un one-shot, mais un pipeline réutilisable.

**Zone d'intérêt (AOI)** : approximation autour de l'épicentre Faenza / Cesena / Forlì / Ravenna, la zone la plus citée dans la couverture presse de l'événement.

In [1]:
import ee
import geemap
from datetime import datetime, timedelta, timezone

PROJECT_ID = "zeta-bonfire-478712-n9"
ee.Initialize(project=PROJECT_ID)

print("GEE prêt")

GEE prêt


In [2]:
# Zone d'intérêt : rectangle englobant Faenza / Cesena / Forlì / Ravenna, Émilie-Romagne
aoi = ee.Geometry.Rectangle([11.55, 44.05, 12.30, 44.45])

Map = geemap.Map()
Map.centerObject(aoi, 10)
Map.addLayer(aoi, {"color": "red"}, "Zone d'intérêt (approximative)")
Map

Map(center=[44.25038667622313, 11.925000000000558], controls=(WidgetControl(options=['position', 'transparent_…

## Charger les images Sentinel-1 avant / après

Même logique que pour la Lettonie : radar (traverse les nuages, utile même si l'événement est ancien on garde la même méthode pour rester cohérent), polarisation VV, avant vs après.

In [3]:
def estimer_prochain_passage(aoi, depuis):
    """Estime la date du prochain passage Sentinel-1 sur l'AOI à partir du cycle de revisite observé récemment."""
    hist = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filterDate(depuis, datetime.now(timezone.utc).strftime("%Y-%m-%d"))
        .sort("system:time_start")
    )
    timestamps = hist.aggregate_array("system:time_start").getInfo()
    dates_uniques = sorted(
        {datetime.fromtimestamp(t / 1000, tz=timezone.utc).date() for t in timestamps}
    )
    if len(dates_uniques) < 2:
        return None, None
    grappes = [[dates_uniques[0]]]
    for d in dates_uniques[1:]:
        if (d - grappes[-1][-1]).days <= 2:
            grappes[-1].append(d)
        else:
            grappes.append([d])
    debuts_grappes = [g[0] for g in grappes]
    ecarts = [
        (debuts_grappes[i + 1] - debuts_grappes[i]).days
        for i in range(len(debuts_grappes) - 1)
    ]
    if not ecarts:
        return None, None
    periode = sorted(ecarts)[len(ecarts) // 2]
    prochain = debuts_grappes[-1]
    aujourdhui = datetime.now(timezone.utc).date()
    while prochain <= aujourdhui:
        prochain += timedelta(days=periode)
    return prochain, periode


s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .select("VV")
)

avant = s1.filterDate("2023-04-15", "2023-05-02")  # avant la première vague (02/05)
apres = s1.filterDate("2023-05-17", "2023-05-24")  # après la seconde vague (16/05)

n_avant = avant.size().getInfo()
n_apres = apres.size().getInfo()

print(f"Images disponibles avant l'événement : {n_avant}")
print(f"Images disponibles après l'événement  : {n_apres}")

donnees_pretes = n_avant > 0 and n_apres > 0

if not donnees_pretes:
    prochain, periode = estimer_prochain_passage(aoi, depuis="2023-04-01")
    print()
    if prochain:
        print(f"Cycle de revisite observé : ~{periode} jours. Prochain passage estimé : {prochain}")
    else:
        print("Historique de passages insuffisant pour estimer la prochaine date.")

Images disponibles avant l'événement : 8
Images disponibles après l'événement  : 4


In [4]:
if donnees_pretes:
    img_avant = avant.median().clip(aoi)
    img_apres = apres.median().clip(aoi)

    vis_params = {"min": -25, "max": 0}

    Map = geemap.Map()
    Map.centerObject(aoi, 10)
    Map.addLayer(img_avant, vis_params, "Avant (VV, dB)")
    Map.addLayer(img_apres, vis_params, "Après (VV, dB)")
    Map
else:
    print("Données incomplètes, voir le message ci-dessus.")

In [5]:
if donnees_pretes:
    difference = img_apres.subtract(img_avant)

    SEUIL_DB = -3
    masque_inondation = difference.lt(SEUIL_DB).selfMask()

    Map = geemap.Map()
    Map.centerObject(aoi, 10)
    Map.addLayer(img_apres, vis_params, "Après (VV, dB)")
    Map.addLayer(masque_inondation, {"palette": ["blue"]}, "Zone probablement inondée")
    Map
else:
    print("Données incomplètes, voir le message ci-dessus.")